# Adult Income Classification


In [1]:
import numpy as np
import pandas as pd
import requests

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
)

In [2]:
!pip install xgboost -q

In [3]:
from xgboost import XGBClassifier

## 1. Load data
Download once, then reuse the local copy.

In [4]:
FILE_NAME = "adult.data"
URL = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"

try:
    with open(FILE_NAME, "r") as f:
        pass
except FileNotFoundError:
    print(f"Downloading {FILE_NAME}...")
    response = requests.get(URL)
    response.raise_for_status()  # raise on 4xx/5xx
    with open(FILE_NAME, "wb") as f:
        f.write(response.content)
    print(f"{FILE_NAME} downloaded successfully.")

COLUMN_NAMES = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country",
    "income",
]

df = pd.read_csv(FILE_NAME, skipinitialspace=True, header=None, names=COLUMN_NAMES)

print(df.shape)
df.info()

(32561, 15)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education-num   32561 non-null  int64 
 5   marital-status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital-gain    32561 non-null  int64 
 11  capital-loss    32561 non-null  int64 
 12  hours-per-week  32561 non-null  int64 
 13  native-country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB


In [5]:
df["income"].value_counts()

,count
income,
<=50K,24720
>50K,7841


## 2. Clean

`?` is this dataset's missing-value marker, not real text. `workclass`, `occupation`,
and `native-country` are the only columns affected. Rows missing `occupation` are
almost always also missing `workclass` (confirmed via the mask below), so dropping
either costs ~2,400 rows total, not ~4,000+.

In [6]:
missing_overlap_mask = (df["occupation"] == "?") & (df["workclass"] == "?")
print("Rows missing both occupation and workclass:", df[missing_overlap_mask].shape)

df = df.replace("?", np.nan)
df = df.dropna()
print("Shape after dropping missing rows:", df.shape)

Rows missing both occupation and workclass: (1836, 15)
Shape after dropping missing rows: (30162, 15)


## 3. Encode

One-hot encode nominal (unordered) categories, so the model doesn't infer a false
ranking between categories (e.g. workclass 3 > 1 would be meaningless). Binary
columns get a direct 0/1 map instead of one-hot, since one-hot would just
duplicate the info.

In [7]:
df = pd.get_dummies(df, columns=[
    "workclass", "marital-status", "occupation",
    "relationship", "race", "native-country",
])

df["sex"] = df["sex"].map({"Male": 1, "Female": 0})
df["income"] = df["income"].map({">50K": 1, "<=50K": 0})

# 'education' is redundant with 'education-num' (same info, ordinal already)
df = df.drop(columns=["education"])

print("Shape after encoding:", df.shape)

Shape after encoding: (30162, 88)


## 4. Split features/target, then train/test split

In [8]:
X = df.drop(columns=["income"])
y = df["income"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, shuffle=True
)

## 5. Scale

Fit only on training data to avoid leaking test-set information into the
transformation.

In [9]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Per-row sample weights to counter class imbalance (~75/25 split).
# Used by LogisticRegression and GradientBoostingClassifier below.
sample_weights = compute_sample_weight(class_weight="balanced", y=y_train)

# scale_pos_weight is XGBoost's own equivalent knob, expressed as a
# single ratio rather than per-row weights.
xgb_scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

## 6. Train + evaluate models

In [10]:
def evaluate(name, y_true, y_pred):
    """Print accuracy, per-class precision/recall/F1, and confusion matrix."""
    print(f"\n=== {name} ===")
    print("Accuracy:", accuracy_score(y_true, y_pred))
    print(classification_report(y_true, y_pred))
    print(confusion_matrix(y_true, y_pred))

In [11]:
model_lr_balanced = LogisticRegression()
model_lr_balanced.fit(X_train_scaled, y_train, sample_weight=sample_weights)
y_pred_lr_balanced = model_lr_balanced.predict(X_test_scaled)
evaluate("Logistic Regression (balanced)", y_test, y_pred_lr_balanced)


=== Logistic Regression (balanced) ===
Accuracy: 0.8120338140228742
              precision    recall  f1-score   support

           0       0.94      0.80      0.86      4503
           1       0.59      0.84      0.69      1530

    accuracy                           0.81      6033
   macro avg       0.76      0.82      0.78      6033
weighted avg       0.85      0.81      0.82      6033

[[3612  891]
 [ 243 1287]]


In [12]:
model_gb = GradientBoostingClassifier(random_state=42)
model_gb.fit(X_train_scaled, y_train, sample_weight=sample_weights)
y_pred_gb = model_gb.predict(X_test_scaled)
evaluate("Gradient Boosting", y_test, y_pred_gb)


=== Gradient Boosting ===
Accuracy: 0.8204873197414222
              precision    recall  f1-score   support

           0       0.95      0.81      0.87      4503
           1       0.60      0.86      0.71      1530

    accuracy                           0.82      6033
   macro avg       0.77      0.83      0.79      6033
weighted avg       0.86      0.82      0.83      6033

[[3630  873]
 [ 210 1320]]


In [13]:
model_xgb = XGBClassifier(
    scale_pos_weight=xgb_scale_pos_weight, random_state=42, eval_metric="logloss"
)
model_xgb.fit(X_train_scaled, y_train)
y_pred_xgb = model_xgb.predict(X_test_scaled)
evaluate("XGBoost", y_test, y_pred_xgb)


=== XGBoost ===
Accuracy: 0.840046411403945
              precision    recall  f1-score   support

           0       0.94      0.83      0.89      4503
           1       0.64      0.86      0.73      1530

    accuracy                           0.84      6033
   macro avg       0.79      0.85      0.81      6033
weighted avg       0.87      0.84      0.85      6033

[[3758  745]
 [ 220 1310]]


## 7. Interpretation

In [14]:
# Gradient Boosting feature importances (magnitude only, not direction)
importances = pd.Series(model_gb.feature_importances_, index=X.columns)
print("Top 15 features by importance (Gradient Boosting):")
print(importances.sort_values(ascending=False).head(15))

Top 15 features by importance (Gradient Boosting):
marital-status_Married-civ-spouse    0.474514
education-num                        0.155537
capital-gain                         0.146551
age                                  0.083966
hours-per-week                       0.042533
capital-loss                         0.032980
occupation_Exec-managerial           0.010709
occupation_Prof-specialty            0.006793
occupation_Farming-fishing           0.006713
occupation_Other-service             0.006699
relationship_Wife                    0.005602
fnlwgt                               0.003405
sex                                  0.003242
marital-status_Never-married         0.002115
relationship_Own-child               0.002048
dtype: float64


In [15]:
# Logistic Regression coefficients (signed -> tells you direction, since
# it's a linear model; features are scaled, so magnitudes are comparable)
coefficients = pd.Series(model_lr_balanced.coef_[0], index=X.columns)
print("Marital-status coefficients (Logistic Regression):")
print(coefficients.filter(like="marital-status_").sort_values(ascending=False))

Marital-status coefficients (Logistic Regression):
marital-status_Married-civ-spouse       0.824738
marital-status_Married-AF-spouse        0.065703
marital-status_Widowed                 -0.100543
marital-status_Married-spouse-absent   -0.102737
marital-status_Separated               -0.152945
marital-status_Divorced                -0.262788
marital-status_Never-married           -0.573462
dtype: float64
